In [1]:
import kasa
import datetime
import sqlite3
import asyncio

In [8]:
ips = {'dishwasher': '192.168.1.2',
       'garage_lights': '192.168.1.19',
       'hottub': '192.168.1.11',
       'pv': '192.168.1.3'}

username="griessbaum@gmail.com"
password="whXzp3RsO$IS!8t"

In [9]:
credentials=kasa.Credentials(
        username=username,
        password=password,
)

In [10]:
import asyncio
import kasa


class KasaEnergyMonitor:
    def __init__(self, host: str, credentials: kasa.Credentials):
        self.host = host
        self.credentials = credentials
        self.device = None
        self.energy_module = None

    async def connect(self):
        """
        Discover and initialize the device.
        Must be called once before polling.
        """
        self.device = await kasa.Discover.discover_single(
            self.host,
            credentials=self.credentials,
        )

        await self.device.update()

        if kasa.Module.Energy not in self.device.modules:
            raise RuntimeError("Device does not support energy monitoring.")

        self.energy_module = self.device.modules[kasa.Module.Energy]

    async def get_realtime(self) -> dict:
        """
        Returns realtime energy data as dict.
        """
        if self.device is None:
            raise RuntimeError("Device not connected. Call connect() first.")

        await self.device.update()
        realtime = await self.energy_module.get_status()

        return {
            "power": realtime.power,
            "voltage": realtime.voltage,
            "current": realtime.current
        }


In [11]:
ip = ips['pv']
monitor = KasaEnergyMonitor(ip, credentials)

await monitor.connect()

In [27]:
monitor.device

<DeviceType.Plug at 192.168.1.3 - pv (EP25)>

In [24]:
data = await monitor.get_realtime()
now = datetime.datetime.now()
data['timestamp'] = now.isoformat()
data

Query failed after successful authentication: Host is 192.168.1.3, Sequence is -1489770279, Response status is 403, Request was {"method":"multipleRequest","request_time_milis":1776097840912,"terminal_uuid":"9T/SffXx/rfcQQAEVs+B8A==","params":{"requests":[{"method":"get_device_time"},{"method":"get_auto_off_config","params":{"start_index":0}},{"method":"get_connect_cloud_state"},{"method":"get_device_info"},{"method":"get_device_usage"}]}}


{'power': 33.473,
 'voltage': 119.85,
 'current': 0.336,
 'timestamp': '2026-04-13T09:30:42.267285'}

In [25]:
conn = sqlite3.connect('power.db')
cursor = conn.cursor()

cursor.execute('''INSERT INTO pv (timestamp, power, voltage, current) VALUES (:timestamp, :power, :voltage, :current)''', data)
conn.commit()
conn.close()